In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_community.chat_models.tongyi import ChatTongyi
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from tiktoken import model

str_Parser=StrOutputParser
#通过RunnableLambda自定义函数将AIMessage转换为指定类型
my_func=RunnableLambda(lambda ai_msg:{"name": ai_msg.content})

model = ChatTongyi(model="qwen3-max")

#第一个提示词模板
first_prompt=PromptTemplate.from_template(
    "我的邻居姓：{lastname},刚生了{gender},请起名，并封装到JSON格式返回给我,"
    "要求key是name,value就是起的名字。请严格尊说格式要求"
)

#第二个提示词模板
second_prompt=PromptTemplate.from_template(
    "姓名:{name},请帮我解析含义。"
)

chain=first_prompt|model|my_func|second_prompt|model|str_Parser

# 也可以直接跳过RunnableLambda类，直接让函数加入链也是可以的
# 因为Runnable接口类在实现_or_的时候，支持Callable接口的实例
# chain=first_prompt|model|(lambda ai_msg:{"name":ai_msg.content})|second_prompt|model|str_parser

res:str=chain.invoke({"lastname":"张","gender":"女儿"})
print(res)
print(type(res))

字节编写lambda匿名函数也可以完成自定义逻辑的数据转换，想怎么转换就怎么转换，更自由。
RunnableLambda类是Langchain内置的，将普通函数等转换为Runnable接口实例，方便自定义函数加入chain。
语法： RunnableLambda(函数对象或者lambda匿名函数)